# Underfitting and overfitting

A model can fail in two opposite ways: it can be too simple to capture the pattern (**underfitting**) or so flexible that it learns the noise of the training data (**overfitting**). This third lecture of block 1 shows both with polynomial regression, explains the **bias–variance trade-off** behind them, and shows how to read training and test error together. Regularisation (ridge, lasso) as a remedy for overfitting follows in Session 7.

```mermaid
flowchart TD
    A["compare training and<br/>test error"] --> B{"training error high?"}
    B -->|"yes"| U["underfitting (high bias):<br/>more features, more flexible model"]
    B -->|"no"| C{"test error much<br/>higher than training?"}
    C -->|"yes"| O["overfitting (high variance):<br/>simpler model, more data,<br/>regularisation (Session 7)"]
    C -->|"no"| G["good fit:<br/>evaluate once on the test set"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Underfitting and overfitting: model complexity

### Concept

**Model complexity** is the flexibility of the family of rules a model can represent. For polynomial regression it is the **degree**: degree 1 is a straight line, degree 2 a parabola, degree 15 a curve with up to 14 bends. A polynomial of degree d in one feature x is a linear regression on the features x, x², …, x^d, created by `PolynomialFeatures`.

- **Underfitting**: the model is too rigid for the pattern. Training error and test error are both high and close to each other.
- **Overfitting**: the model follows the noise of the training sample. Training error is low, test error is much higher.
- In between lies the complexity with the lowest test error.

![Polynomial fits of degree 1, 4 and 15 to 20 noisy points from a cosine curve, and training versus test error by degree](../figures/under-overfitting.png)

The degree-1 line misses the curve (underfit). Degree 4 follows it. Degree 15 bends towards individual points and shoots off at the right edge, where there are few data (overfit). The right panel shows the typical pattern: training error falls steadily with complexity; test error falls, reaches a minimum, and rises again.

### Why it matters

Every flexible model, from decision trees to neural networks, can overfit. The gap between training and test error is the first diagnostic in every later session. Choosing the complexity on the test set itself would make the test score optimistic, which is why Session 7 introduces cross-validation.

### How it works in Python

In [2]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(0)
x_train = np.sort(rng.uniform(0, 1, 20)).reshape(-1, 1)        # 20 noisy training points
y_train = np.cos(1.5 * np.pi * x_train.ravel()) + rng.normal(0, 0.15, 20)
x_test = rng.uniform(0, 1, 1000).reshape(-1, 1)                # new data from the same source
y_test = np.cos(1.5 * np.pi * x_test.ravel()) + rng.normal(0, 0.15, 1000)

for degree in [1, 2, 4, 8, 15]:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression()).fit(x_train, y_train)
    train_rmse = root_mean_squared_error(y_train, model.predict(x_train))
    test_rmse = root_mean_squared_error(y_test, model.predict(x_test))
    print(f"degree {degree:2d}  train {train_rmse:.3f}  test {test_rmse:.3f}")
# degree  1  train 0.449  test 0.459
# degree  2  train 0.220  test 0.257
# degree  4  train 0.084  test 0.169
# degree  8  train 0.069  test 0.329
# degree 15  train 0.061  test 0.650

degree  1  train 0.449  test 0.459
degree  2  train 0.220  test 0.257
degree  4  train 0.084  test 0.169
degree  8  train 0.069  test 0.329
degree 15  train 0.061  test 0.650


On the Berlin listings, model the log price with a polynomial in two features, the number of guests and the distance to Alexanderplatz. Price does not fall linearly with distance (the centre is a plateau, the outskirts flatten out), so some curvature should help. Compare all 5,360 training listings with a training set of only 60:

In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
short = listings[listings["price"].notna() & listings["minimum_nights"].lt(28)].copy()
short["log_price"] = np.log(short["price"])
short["km_to_centre"] = 111.2 * np.hypot(short["latitude"] - 52.5219,
                                         (short["longitude"] - 13.4132) * np.cos(np.radians(52.52)))
features = ["accommodates", "km_to_centre"]
train, test = train_test_split(short, test_size=0.2, random_state=42)
small = train.sample(60, random_state=0)

for degree in [1, 2, 3, 5, 8]:
    for name, part in [("5,360", train), ("60", small)]:
        model = make_pipeline(StandardScaler(), PolynomialFeatures(degree), LinearRegression())
        model.fit(part[features], part["log_price"])
        print(f"degree {degree}, {name:>5s} rows: train RMSE "
              f"{root_mean_squared_error(part['log_price'], model.predict(part[features])):6.3f}"
              f"  test RMSE {root_mean_squared_error(test['log_price'], model.predict(test[features])):9.3f}")

degree 1, 5,360 rows: train RMSE  0.468  test RMSE     0.471
degree 1,    60 rows: train RMSE  0.415  test RMSE     0.475
degree 2, 5,360 rows: train RMSE  0.453  test RMSE     0.454
degree 2,    60 rows: train RMSE  0.380  test RMSE     0.466
degree 3, 5,360 rows: train RMSE  0.449  test RMSE     0.452
degree 3,    60 rows: train RMSE  0.373  test RMSE     0.538
degree 5, 5,360 rows: train RMSE  0.447  test RMSE     0.452
degree 5,    60 rows: train RMSE  0.328  test RMSE     2.018
degree 8, 5,360 rows: train RMSE  0.445  test RMSE     0.457
degree 8,    60 rows: train RMSE  0.193  test RMSE  1655.897


```
degree 1, 5,360 rows: train RMSE  0.468  test RMSE     0.471
degree 1,    60 rows: train RMSE  0.415  test RMSE     0.475
degree 2, 5,360 rows: train RMSE  0.453  test RMSE     0.454
degree 2,    60 rows: train RMSE  0.380  test RMSE     0.466
degree 3, 5,360 rows: train RMSE  0.449  test RMSE     0.452
degree 3,    60 rows: train RMSE  0.373  test RMSE     0.538
degree 5, 5,360 rows: train RMSE  0.447  test RMSE     0.452
degree 5,    60 rows: train RMSE  0.328  test RMSE     2.018
degree 8, 5,360 rows: train RMSE  0.445  test RMSE     0.457
degree 8,    60 rows: train RMSE  0.193  test RMSE  1655.897
```

With 5,360 training listings, a little curvature helps (test RMSE from 0.471 to 0.452 at degree 3) and more does not: from degree 5 on, training and test error stay close and the test error creeps up again. With 60 training listings, the training error falls steadily (to 0.193 at degree 8) while the test error rises from degree 3 on and explodes at degree 8, where the polynomial predicts absurd prices for listings outside the range of the 60 it has seen. Overfitting is a problem of **flexibility relative to the amount of data**. Note also how much remains unexplained: an RMSE of 0.45 on the log scale means typical errors of a factor of about e^0.45 ≈ 1.6. Guests and distance are not enough; room type and district ([lecture 01](01-lecture-lifecycle-and-linear-regression.ipynb)) help more than any polynomial.

### In practice

- Google Flu Trends (2008–2015) fitted search-term frequencies to past flu data and later overestimated flu prevalence substantially, a widely cited case of a model fitted to past patterns that did not generalise (Lazer et al., 2014).
- Clinical prediction models developed on small cohorts often perform worse at external validation in other hospitals; reporting guidelines such as TRIPOD require validation on separate data.
- Finance: trading strategies backtested over many variants look profitable on past data and fail on new data ("backtest overfitting", Bailey et al., 2014).

> **Important.** **Worked example.** Section 1.5 of the case-study notebook [04-case-study-airbnb-price-and-churn.ipynb](04-case-study-airbnb-price-and-churn.ipynb) compares training and test error for increasing polynomial degree, first on the toy data, then on the Berlin price model. The scikit-learn example [05-underfitting-overfitting.ipynb](05-underfitting-overfitting.ipynb) shows degrees 1, 4 and 15 on a cosine curve with cross-validated errors.

> 📖 **Read more:** [extras/05-polynomial-features.ipynb](extras/05-polynomial-features.ipynb) (INRIA scikit-learn MOOC): a linear model on polynomial features, binned features and spline features fits non-linear relationships while staying linear in its parameters.

## The bias–variance trade-off

### Concept

The expected test error of a model at a point can be split into three parts:

expected squared error = **bias²** + **variance** + **irreducible noise**.

- **Bias** is the systematic error of the model family: how far the average prediction (over many possible training samples) is from the truth. A straight line fitted to a curve has high bias, however much data it gets.
- **Variance** is how much the prediction changes from one training sample to another. A degree-15 polynomial fitted to 20 points changes drastically when the points change.
- **Noise** is the variation in y that no model using these features can predict.

More complexity lowers bias and raises variance. The best complexity balances the two. More training data lowers variance, so a larger dataset can support a more complex model.

```mermaid
xychart-beta
    title "Error components by model complexity (schematic)"
    x-axis "model complexity" [1, 2, 3, 4, 5, 6, 7, 8]
    y-axis "error" 0 --> 10
    line "bias squared" [8, 5, 3.2, 2, 1.3, 0.9, 0.6, 0.5]
    line "variance" [0.3, 0.5, 0.8, 1.3, 2, 3, 4.3, 6]
    line "total test error" [9.3, 6.5, 5, 4.3, 4.3, 4.9, 5.9, 7.5]
```

Worked example: suppose the true relationship is a curve and you fit straight lines to 100 different samples. All 100 lines miss the curve in the same way (bias) but are similar to each other (low variance). Fit degree-15 polynomials instead: on average they follow the curve (low bias) but each one wiggles differently (high variance).

### Why it matters

The trade-off explains why "use the most flexible model" is not a strategy, and why the remedies differ: underfitting needs more flexibility or better features; overfitting needs less flexibility, more data or regularisation (Session 7). Ensembles (Session 10) reduce variance by averaging many models.

### How it works in Python

Estimate bias and variance by refitting on many training samples:

In [4]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(1)
x_grid = np.linspace(0.05, 0.95, 50).reshape(-1, 1)
truth = np.cos(1.5 * np.pi * x_grid.ravel())

for degree in [1, 4, 12]:
    preds = []
    for _ in range(200):                                       # 200 different training samples
        x = rng.uniform(0, 1, 20).reshape(-1, 1)
        y = np.cos(1.5 * np.pi * x.ravel()) + rng.normal(0, 0.15, 20)
        model = make_pipeline(PolynomialFeatures(degree), LinearRegression()).fit(x, y)
        preds.append(model.predict(x_grid))
    preds = np.array(preds)
    bias2 = np.mean((preds.mean(axis=0) - truth) ** 2)
    variance = np.mean(preds.var(axis=0))
    print(f"degree {degree:2d}  bias^2 {bias2:.3f}  variance {variance:.3f}")
# degree  1  bias^2 0.145  variance 0.029
# degree  4  bias^2 0.000  variance 0.008
# degree 12  bias^2 0.013  variance 7.851

degree  1  bias^2 0.145  variance 0.029


degree  4  bias^2 0.000  variance 0.008


degree 12  bias^2 0.013  variance 7.851


### In practice

- Netflix Prize (2006–2009): the winning solutions blended hundreds of models, reducing variance by averaging.
- Weather and climate services run **ensembles** of forecasts with perturbed starting conditions to quantify forecast variance.
- Credit-risk modelling in banks favours simple, stable scorecards partly because regulators require predictions that do not change erratically with new data.

> **Note.** The decomposition is exact for squared error. For other losses (classification error, log-loss) the idea carries over, but the formula differs.

## Check your understanding

1. A model has training RMSE 0.06 and test RMSE 0.65. Underfitting or overfitting? Name two remedies.
2. Why does the degree-8 polynomial hardly overfit on the 5,360 training listings, although it fails completely on 60?
3. Explain bias and variance with the example of a straight line and a degree-15 polynomial fitted to a curve.

## Further reading

- James, G., et al. (2023). *An Introduction to Statistical Learning with Applications in Python*, section 2.2 "Assessing model accuracy" (bias–variance trade-off). Springer. <https://www.statlearning.com/>
- scikit-learn developers (2026). *Underfitting vs. overfitting* (example). <https://scikit-learn.org/stable/auto_examples/model_selection/plot_underfitting_overfitting.html>